In [1]:
import sqlite3
import pandas as pd

In [2]:
df = pd.read_csv("D:\PowerBI Projects\Pharma_Dashboard\Data\cleaned_pharma_sales.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_8356\2157900171.py:1: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_csv("D:\PowerBI Projects\Pharma_Dashboard\Data\cleaned_pharma_sales.csv")


,Distributor,Customer Name,City,Country,Latitude,Longitude,Channel,Sub-channel,Product Name,Product Class,Quantity,Price,Sales,Month,Year,Name of Sales Rep,Manager,Sales Team
0,Gottlieb-Cruickshank,"Zieme, Doyle and Kunze",Lublin,Poland,51.2333,22.5667,Hospital,Private,Topipizole,Mood Stabilizers,4.0,368,1472.0,January,2018,Mary Gerrard,Britanny Bold,Delta
1,Gottlieb-Cruickshank,Feest PLC,Świecie,Poland,53.4167,18.4333,Pharmacy,Retail,Choriotrisin,Antibiotics,7.0,591,4137.0,January,2018,Jessica Smith,Britanny Bold,Delta
2,Gottlieb-Cruickshank,Medhurst-Beer Pharmaceutical Limited,Rybnik,Poland,50.0833,18.5000,Pharmacy,Institution,Acantaine,Antibiotics,30.0,66,1980.0,January,2018,Steve Pepple,Tracy Banks,Bravo
3,Gottlieb-Cruickshank,Barton Ltd Pharma Plc,Czeladź,Poland,50.3333,19.0833,Hospital,Private,Lioletine Refliruvax,Analgesics,6.0,435,2610.0,January,2018,Mary Gerrard,Britanny Bold,Delta
4,Gottlieb-Cruickshank,Keeling LLC Pharmacy,Olsztyn,Poland,53.7800,20.4942,Pharmacy,Retail,Oxymotroban Fexoformin,Analgesics,20.0,458,9160.0,January,2018,Anne Wu,Britanny Bold,Delta


In [3]:
conn = sqlite3.connect("pharma.db")

df.to_sql(
    "pharma_sales",
    conn,
    if_exists="replace",
    index=False
)

print("Database created successfully!")

Database created successfully!


In [4]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,pharma_sales


# Customer Performance Analysis

Objective:
Identify the highest-value customers based on revenue, purchase quantity, and engagement.


In [5]:
query = """
SELECT
    [Customer Name],
    ROUND(SUM(Sales),2) AS Total_Sales,
    SUM(Quantity) AS Total_Quantity,
    ROUND(AVG(Price),2) AS Average_Price,
    COUNT(DISTINCT Year || '-' || Month) AS Active_Months
FROM pharma_sales
GROUP BY [Customer Name]
ORDER BY Total_Sales DESC
LIMIT 20;
"""

customer_df = pd.read_sql(query, conn)

customer_df

,Customer Name,Total_Sales,Total_Quantity,Average_Price,Active_Months
0,Mraz-Kutch Pharma Plc,93561780.00,174234.000000,408.84,48
1,"Parker, Green and Emmerich Pharma Plc",64890501.00,132080.000000,422.93,48
2,Zemlak-Witting,56006680.00,124868.000000,408.16,48
3,"Moen, Murazik and Smith Pharm",50885320.00,94970.000000,410.79,47
4,Mills Inc Pharmaceutical Ltd,45939009.00,89970.000000,424.83,48
5,Streich PLC,44454616.00,103978.000000,413.95,48
6,"Gleichner, Bahringer and Morar Pharmaceutical ...",43779381.00,100146.000000,411.37,48
7,Reichel Inc,43495626.00,90563.000000,411.79,48
8,"Romaguera, Moen and Hagenes Pharmacy",42066659.97,91253.734824,419.83,48
9,"Torphy, Pfeffer and Jakubowski Pharm",41514215.00,85885.000000,399.18,48


In [6]:
print("Business Insight")

print(
    """
Top customers contribute a significant portion of revenue.
These accounts should be prioritized for long-term relationship management
and targeted commercial strategies.
"""
)

Business Insight

Top customers contribute a significant portion of revenue.
These accounts should be prioritized for long-term relationship management
and targeted commercial strategies.



# Sales Representative Performance

Objective:
Evaluate individual sales representatives based on revenue generated and customer coverage.


In [7]:
query = """
SELECT
    [Name of Sales Rep],
    ROUND(SUM(Sales),2) AS Total_Sales,
    SUM(Quantity) AS Total_Quantity,
    COUNT(DISTINCT [Customer Name]) AS Customers_Handled
FROM pharma_sales
GROUP BY [Name of Sales Rep]
ORDER BY Total_Sales DESC;
"""

rep_df = pd.read_sql(query, conn)

rep_df

,Name of Sales Rep,Total_Sales,Total_Quantity,Customers_Handled
0,Jimmy Grey,9.859700e+08,2.293164e+06,751
1,Abigail Thompson,9.810570e+08,2.301505e+06,751
2,Sheila Stones,9.582039e+08,2.384530e+06,751
3,Daniel Gates,9.506586e+08,2.310476e+06,751
4,Anne Wu,9.201683e+08,2.230109e+06,751
5,Morris Garcia,9.011955e+08,2.255844e+06,751
6,Stella Given,8.883409e+08,2.109224e+06,751
7,Jessica Smith,8.816976e+08,2.143397e+06,751
8,Steve Pepple,8.754500e+08,2.237292e+06,751
9,Mary Gerrard,8.752669e+08,2.111203e+06,751


In [8]:
print(
"""
High-performing sales representatives consistently generate greater
revenue while managing a larger customer portfolio. This information
can support incentive planning and resource allocation.
"""
)


High-performing sales representatives consistently generate greater
revenue while managing a larger customer portfolio. This information
can support incentive planning and resource allocation.



# Sales Team Performance

Objective:
Compare sales teams to identify top-performing groups.



In [9]:
query = """
SELECT
    [Sales Team],
    ROUND(SUM(Sales),2) AS Total_Sales,
    SUM(Quantity) AS Total_Quantity,
    COUNT(DISTINCT [Customer Name]) AS Customers
FROM pharma_sales
GROUP BY [Sales Team]
ORDER BY Total_Sales DESC;
"""

team_df = pd.read_sql(query, conn)

team_df

,Sales Team,Total_Sales,Total_Quantity,Customers
0,Delta,3.635337e+09,8.869239e+06,751
1,Charlie,2.824970e+09,6.712863e+06,751
2,Bravo,2.757702e+09,6.794641e+06,751
3,Alfa,2.580969e+09,6.301975e+06,751


# Manager Analysis

Objective:
Determine which Manager helped most to contribute the highest revenue.

In [10]:
query = """
SELECT
    Manager,
    ROUND(SUM(Sales),2) AS Team_Sales,
    COUNT(DISTINCT [Name of Sales Rep]) AS Sales_Reps
FROM pharma_sales
GROUP BY Manager
ORDER BY Team_Sales DESC;
"""

manager_df = pd.read_sql(query, conn)

manager_df

,Manager,Team_Sales,Sales_Reps
0,Britanny Bold,3.635337e+09,4
1,Alisha Cordwell,2.824970e+09,3
2,Tracy Banks,2.757702e+09,3
3,James Goodwill,2.580969e+09,3


# Product Class Analysis

Objective:
Determine which pharmaceutical product classes contribute the highest revenue.


In [11]:
query = """
SELECT
    [Product Class],
    ROUND(SUM(Sales),2) AS Total_Sales
FROM pharma_sales
GROUP BY [Product Class]
ORDER BY Total_Sales DESC;
"""

product_df = pd.read_sql(query, conn)

product_df

,Product Class,Total_Sales
0,Analgesics,2.371510e+09
1,Antiseptics,2.237520e+09
2,Mood Stabilizers,2.058910e+09
3,Antipiretics,1.883306e+09
4,Antibiotics,1.750277e+09
5,Antimalarial,1.497455e+09


In [12]:
conn.close()

print("SQLite connection closed.")

SQLite connection closed.
